# Exercise 1: Machine Learning Basics

**Unit:** Unit 3: ML Basics — regression vs classification, the XOR problem,
gradient descent, and model interpretability.

## 📝 Instructions

Each exercise gives you a **working baseline** you can run immediately, followed by a
**🖊️ Your turn** extension marked with `# TODO`. The notebook runs clean top-to-bottom
before you change anything — your job is to extend it.

**Steps:**
1. Run each baseline cell and read its output.
2. Complete the 🖊️ Your turn tasks in the same cell.
3. Re-run and check your results.
4. Solutions are released by your instructor.

---

## Exercise 1: Regression vs Classification

Same framing as `01_regression_classification.ipynb`: regression predicts a **number**,
classification predicts a **category** — and each has its own metrics.

In [1]:
# Exercise 1: Regression vs classification on synthetic data
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import mean_squared_error, accuracy_score

print("Exercise 1: Regression vs Classification")
print("-" * 60)

rng = np.random.default_rng(0)

# --- Regression: predict exam score from hours studied ---
hours = rng.uniform(0, 10, size=(150, 1))
score = 8 * hours.ravel() + 10 + rng.normal(0, 6, 150)

X_train, X_test, y_train, y_test = train_test_split(hours, score, test_size=0.2, random_state=0)
reg = LinearRegression().fit(X_train, y_train)
mse = mean_squared_error(y_test, reg.predict(X_test))
print(f"  Regression  — test MSE: {mse:.1f}, test R^2: {reg.score(X_test, y_test):.3f}")

# --- Classification: pass/fail (score >= 50) from hours studied ---
passed = (score >= 50).astype(int)
Xc_train, Xc_test, yc_train, yc_test = train_test_split(hours, passed, test_size=0.2, random_state=0)
clf = LogisticRegression().fit(Xc_train, yc_train)
acc = accuracy_score(yc_test, clf.predict(Xc_test))
print(f"  Classification — test accuracy: {acc:.3f}")

# 🖊️ Your turn:
# TODO 1: Print one regression prediction (e.g. reg.predict([[7.5]])) and one
#         classification prediction for the same 7.5 hours. How do the two
#         answers differ in KIND, not just value?
# TODO 2: Increase the noise (rng.normal(0, 20, 150)) and re-run. Which metric
#         degrades more, MSE or accuracy? Why?


Exercise 1: Regression vs Classification
------------------------------------------------------------
  Regression  — test MSE: 45.2, test R^2: 0.906
  Classification — test accuracy: 0.933


## Exercise 2: The XOR Problem and Linear Separability

A **linear** model cannot separate XOR (`02_perceptron_xor.ipynb`). But watch what a
single hand-crafted feature does.

In [2]:
# Exercise 2: XOR — a linear model fails, a crafted feature fixes it
import numpy as np
from sklearn.linear_model import LogisticRegression

print("Exercise 2: XOR and Linear Separability")
print("-" * 60)

X_xor = np.array([[0, 0], [0, 1], [1, 0], [1, 1]])
y_xor = np.array([0, 1, 1, 0])

# Baseline: a purely linear model on the raw inputs
# (C=100 weakens sklearn's default regularization, which otherwise
#  keeps the weights too small for a clean fit on just 4 points)
linear_clf = LogisticRegression(C=100).fit(X_xor, y_xor)
linear_acc = linear_clf.score(X_xor, y_xor)
print(f"  Linear model on raw (x1, x2):        accuracy = {linear_acc:.2f}")
print("  A single line cannot put (0,1) and (1,0) on one side and")
print("  (0,0) and (1,1) on the other - XOR is not linearly separable.")

# Baseline: add the product feature x1*x2 and try again
X_feat = np.column_stack([X_xor, X_xor[:, 0] * X_xor[:, 1]])
feat_clf = LogisticRegression(C=100).fit(X_feat, y_xor)
feat_acc = feat_clf.score(X_feat, y_xor)
print(f"\n  Same model + crafted feature x1*x2:  accuracy = {feat_acc:.2f}")
print("  The extra feature bends the space so one line suffices -")
print("  hidden layers in a neural network learn such features automatically")
print("  (that is what 03_solving_xor_keras.ipynb does).")

# 🖊️ Your turn:
# TODO 1: Try a different crafted feature: (x1 - x2)**2. Does it also reach 1.00?
# TODO 2: Print the model predictions for all four XOR rows in both versions
#         (linear_clf needs the 2-column input, feat_clf the 3-column one).


Exercise 2: XOR and Linear Separability
------------------------------------------------------------
  Linear model on raw (x1, x2):        accuracy = 0.50
  A single line cannot put (0,1) and (1,0) on one side and
  (0,0) and (1,1) on the other - XOR is not linearly separable.

  Same model + crafted feature x1*x2:  accuracy = 1.00
  The extra feature bends the space so one line suffices -
  hidden layers in a neural network learn such features automatically
  (that is what 03_solving_xor_keras.ipynb does).


## Exercise 3: Gradient Descent by Hand

Like `04_gradient_descent_loss_functions.ipynb`: fit `y = w * x` by repeatedly stepping
`w` against the gradient of the MSE loss.

In [3]:
# Exercise 3: Gradient descent for a one-parameter model y = w * x
import numpy as np

print("Exercise 3: Gradient Descent")
print("-" * 60)

rng = np.random.default_rng(1)
x = rng.uniform(0, 5, 80)
y = 3.0 * x + rng.normal(0, 1, 80)   # true w is 3.0

def mse_loss(w):
    return np.mean((y - w * x) ** 2)

def gradient(w):
    # d/dw of mean((y - w*x)^2) = -2 * mean(x * (y - w*x))
    return -2 * np.mean(x * (y - w * x))

w = 0.0            # start far from the truth
learning_rate = 0.02

print(f"  {'step':>4s} {'w':>8s} {'loss':>10s}")
for step in range(31):
    if step % 5 == 0:
        print(f"  {step:4d} {w:8.4f} {mse_loss(w):10.4f}")
    w = w - learning_rate * gradient(w)

print(f"\n  Final w = {w:.4f} (true value 3.0) - the loss shrank at every printed step.")

# 🖊️ Your turn:
# TODO 1: Set learning_rate = 0.2 and re-run. What happens to the loss column?
# TODO 2: Set learning_rate = 0.001. How many steps would you now need, roughly?
# TODO 3: Add a bias term (y = w*x + b) and update both w and b each step.


Exercise 3: Gradient Descent
------------------------------------------------------------
  step        w       loss
     0   0.0000    84.1539
     5   2.6994     1.6820
    10   2.9622     0.9000
    15   2.9878     0.8926
    20   2.9903     0.8925
    25   2.9906     0.8925
    30   2.9906     0.8925

  Final w = 2.9906 (true value 3.0) - the loss shrank at every printed step.


## Exercise 4: Interpretability — Which Features Matter?

`05_model_interpretability_shap_lime.ipynb` used SHAP and LIME; here you check the
simplest interpretability signal — impurity-based feature importances — on data where
**you know the ground truth**.

In [4]:
# Exercise 4: Feature importances on data with known informative features
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

print("Exercise 4: Interpretability")
print("-" * 60)

rng = np.random.default_rng(7)
n = 400

# Two informative features and two pure-noise features
informative_1 = rng.normal(size=n)
informative_2 = rng.normal(size=n)
noise_1 = rng.normal(size=n)
noise_2 = rng.normal(size=n)

X = np.column_stack([informative_1, informative_2, noise_1, noise_2])
y = (informative_1 + 0.5 * informative_2 > 0).astype(int)   # label ignores the noise
feature_names = ["informative_1", "informative_2", "noise_1", "noise_2"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=7)
forest = RandomForestClassifier(n_estimators=100, random_state=7).fit(X_train, y_train)
print(f"  Test accuracy: {forest.score(X_test, y_test):.3f}\n")

print("  Impurity-based feature importances (should rank the informative pair on top):")
order = np.argsort(forest.feature_importances_)[::-1]
for idx in order:
    print(f"    {feature_names[idx]:14s} {forest.feature_importances_[idx]:.3f}")

# 🖊️ Your turn:
# TODO 1: Make the label depend on noise_1 as well (e.g. + 0.5 * noise_1) and re-run.
#         Does its importance rise?
# TODO 2: Add a fifth feature that is a COPY of informative_1. What happens to the
#         importance previously assigned to informative_1, and why is that a warning
#         about reading importances too literally?


Exercise 4: Interpretability
------------------------------------------------------------
  Test accuracy: 0.970

  Impurity-based feature importances (should rank the informative pair on top):
    informative_1  0.690
    informative_2  0.226
    noise_1        0.043
    noise_2        0.041


---

## ✅ Check Your Work

- ✅ Does the whole notebook still run top-to-bottom after your edits?
- ✅ Exercise 2: can you say in one sentence why the crafted feature fixes XOR?
- ✅ Exercise 3: can you explain what a too-large learning rate did to the loss?
- ✅ Exercise 4: did the importance ranking match what you built into the data?

**Next steps:**
- Solutions are released by your instructor.
- Take the quiz in `../quizzes/` and continue to Unit 4: `../../unit4-neural-networks-basics/README.md`.